# FUT.AI | Homografía con clics — versión fácil (Google Colab)

## ¿Qué vas a hacer?

Marcarás un lugar conocido de la cancha **dos veces**: primero en la foto del partido y después en el mapa visto desde arriba. Repite al menos **4 pares**. Al final aparecerán los jugadores sobre el mapa de cancha.

**Importante:** las imágenes de la interfaz aparecen **después de ejecutar la celda 3**; estas sí responden a los clics. **No necesitas hacer clic sobre imágenes de salida normal de Matplotlib.**

**Preparación:** asegúrate de tener cargados en Colab el MP4 y el CSV que generaste con YOLO11s + ByteTrack. A continuación, ejecuta las celdas **1 → 2 → 3 → 4 → 5 → 6**, en orden.

**Restricción:** esta versión calcula la geometría **para un solo fotograma**; no mide velocidades en clips con cámara móvil.


## Celda 1 — Abre el video y el archivo de IDs

Si Colab indica que no encuentra el MP4 o CSV, mira el explorador de archivos de la izquierda y actualiza sus rutas aquí. Puedes cambiar `SEGUNDO_CALIBRACION` para buscar un instante con suficientes líneas de la cancha.

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# CAMBIA ESTAS RUTAS para que coincidan con tu sesión de Colab
VIDEO = Path('/content/prueba2_20s.mp4')
CSV_TRACKS = Path('/content/futai/diagnostico/prueba2_20s_yolo11s.csv')
SEGUNDO_CALIBRACION = 5.0  # Cambia el segundo hasta ver >=4 referencias conocidas
LARGO_M = 105.0
ANCHO_M = 68.0
SALIDA = Path('/content/futai/homografia')
SALIDA.mkdir(parents=True, exist_ok=True)

if not VIDEO.is_file():
    raise FileNotFoundError(f'Falta el video: {VIDEO}. Súbelo a Colab o corrige VIDEO.')
if not CSV_TRACKS.is_file():
    raise FileNotFoundError(f'Falta el CSV: {CSV_TRACKS}. Corrige CSV_TRACKS.')

tracks = pd.read_csv(CSV_TRACKS)
columnas = {'frame','tiempo_s','track_id','confianza','x1','y1','x2','y2'}
if not columnas.issubset(tracks.columns):
    raise ValueError(f'Columnas faltantes: {columnas-set(tracks.columns)}')

cap = cv2.VideoCapture(str(VIDEO))
if not cap.isOpened():
    raise ValueError('El video no se puede abrir')
fps = float(cap.get(cv2.CAP_PROP_FPS))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
if fps <= 0:
    cap.release()
    raise ValueError('FPS inválido')
frame_idx = max(0, min(round(SEGUNDO_CALIBRACION*fps), total_frames-1))
cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
ok, frame_bgr = cap.read()
cap.release()
if not ok:
    raise ValueError('No se pudo leer el fotograma elegido')

alto_original, ancho_original = frame_bgr.shape[:2]
if tracks[['x2','y2']].max().iloc[0] > ancho_original+5 or tracks[['x2','y2']].max().iloc[1] > alto_original+5:
    raise ValueError('CSV y video no tienen el mismo sistema de coordenadas. Verifica que correspondan al mismo archivo.')
print(f'Video: {VIDEO.name} | {ancho_original}x{alto_original} | {fps:.2f} FPS')
print(f'Calibración: frame {frame_idx}, t = {frame_idx/fps:.2f} segundos')
print(f'CSV: {len(tracks):,} observaciones, {tracks.track_id.nunique()} tracks')

## Celda 2 — Preparar el mapa de la cancha

Esta celda prepara el plano, pero **NO muestra imágenes para hacer clic**. El panel interactivo aparecerá en la próxima celda.

In [ ]:
# Plano esquemático de referencia, en escala métrica (no una medición del estadio).
def construir_plano(largo, ancho, escala=6, margen=42):
    w, h = int(round(largo*escala+2*margen)), int(round(ancho*escala+2*margen))
    im = np.full((h,w,3), (44,116,63), dtype=np.uint8)

    def p(x,y):
        return (int(round(margen+x*escala)), int(round(margen+y*escala)))
    blanco=(245,245,245)
    cv2.rectangle(im, p(0,0), p(largo,ancho), blanco, 2)
    cv2.line(im,p(largo/2,0),p(largo/2,ancho),blanco,2)
    cv2.circle(im,p(largo/2,ancho/2),int(round(9.15*escala)),blanco,2)
    cv2.circle(im,p(largo/2,ancho/2),4,blanco,-1)
    for izq in (True,False):
        x_area = 16.5 if izq else largo-16.5
        x_meta = 0 if izq else largo
        y1,y2=(ancho-40.32)/2,(ancho+40.32)/2
        cv2.rectangle(im,p(min(x_meta,x_area),y1),p(max(x_meta,x_area),y2),blanco,2)
        x_chica = 5.5 if izq else largo-5.5
        ym1,ym2=(ancho-18.32)/2,(ancho+18.32)/2
        cv2.rectangle(im,p(min(x_meta,x_chica),ym1),p(max(x_meta,x_chica),ym2),blanco,2)
        x_pen = 11 if izq else largo-11
        cv2.circle(im,p(x_pen,ancho/2),3,blanco,-1)
    # Ejes: x a lo largo del campo desde la portería izquierda, y de banda superior a inferior.
    cv2.putText(im, '0 m',p(0,0),cv2.FONT_HERSHEY_SIMPLEX,0.45,(255,255,255),1)
    cv2.putText(im, f'{largo:g} m',p(largo-10,0),cv2.FONT_HERSHEY_SIMPLEX,0.45,(255,255,255),1)
    return im, escala, margen

plano_bgr, ESCALA_PLANO, MARGEN_PLANO = construir_plano(LARGO_M,ANCHO_M)
print('✅ Mapa preparado. EJECUTA la siguiente celda para seleccionar tus puntos con el mouse.')

## Celda 3 — AQUÍ SÍ HACES CLIC (panel interactivo)

Al ejecutarla, debe aparecer un **panel oscuro** con dos imágenes y los botones **Deshacer**, **Reiniciar**, **Terminar**.

1. **Haz clic en una intersección de líneas** de la **imagen superior (video)**. Aparece un número **1**.
2. Haz clic en **la misma intersección física** en la **imagen inferior (mapa)**. Aparece el número **1** allí también.
3. Repite con 4 o más lugares distintos: aparecerán **2, 3, 4...**. Evita seleccionar cuatro puntos de una misma línea.
4. Pulsa **✅ Terminar y guardar puntos**. Solo ahora pasamos a calcular la homografía.

**Buenos puntos:** intersección de la mitad de cancha con la banda, cruces de las líneas del área con sus líneas perpendiculares, centro marcado del campo. **Malos puntos:** jugadores, publicidad, partes arbitrarias de una línea, intersecciones que no aparecen realmente en el video. **Si no ves cuatro referencias útiles en ese fotograma, cambia de segundo en la celda 1 y vuelve a ejecutar desde allí.**

> Si únicamente ves imágenes inmóviles sin encabezados ni botones, **no se ejecutó esta celda**. Si el panel oscuro sí aparece pero el clic no genera marcadores, avísame y envíame una captura del panel para usar un método alternativo.


In [ ]:
from IPython.display import display
from google.colab.output import eval_js
import base64

# Redimensionamos SOLO para la interfaz; los clics se convierten a los píxeles originales.
ESCALA_VISUAL = min(1.0, 1100/ancho_original)
frame_muestra = cv2.resize(frame_bgr, (round(ancho_original*ESCALA_VISUAL), round(alto_original*ESCALA_VISUAL)))
_, buf_frame = cv2.imencode('.jpg',frame_muestra,[cv2.IMWRITE_JPEG_QUALITY,90])
_, buf_plano = cv2.imencode('.png',plano_bgr)
url_frame = 'data:image/jpeg;base64,'+base64.b64encode(buf_frame).decode('ascii')
url_plano = 'data:image/png;base64,'+base64.b64encode(buf_plano).decode('ascii')

js = r"""new Promise((resolve, reject) => {
  const VIDEO_URL = "__URL_FRAME__";
  const PITCH_URL = "__URL_PLANO__";
  const ORIGINAL_W = __ORIGINAL_W__;
  const ORIGINAL_H = __ORIGINAL_H__;
  const PAD = __PAD__;
  const SCALE = __SCALE__;
  const LARGO = __LARGO__;
  const ANCHO = __ANCHO__;
  const root = document.createElement('div');
  root.style = 'font:15px Arial,sans-serif;max-width:1110px;background:#152b28;color:#fff;padding:18px;border:3px solid #4ade80;border-radius:12px;box-sizing:border-box;';
  const title=document.createElement('h2');title.textContent='⚽ FUT.AI — PANEL PARA MARCAR PUNTOS';root.appendChild(title);
  const estado=document.createElement('p');estado.style='background:#314b3d;padding:12px;border-radius:8px;font-weight:bold;';root.appendChild(estado);
  const lab1=document.createElement('h3');lab1.textContent='1. VIDEO DEL PARTIDO (HAZ CLIC AQUÍ PRIMERO)';root.appendChild(lab1);
  const cv=document.createElement('canvas');root.appendChild(cv);
  const lab2=document.createElement('h3');lab2.textContent='2. MAPA DE LA CANCHA (DESPUÉS HAZ CLIC EN EL MISMO PUNTO)';root.appendChild(lab2);
  const cp=document.createElement('canvas');root.appendChild(cp);
  for(const c of [cv,cp]){c.style='display:block;width:auto;max-width:100%;height:auto;cursor:crosshair;border:2px solid #e8f6ea;margin:10px 0 14px;box-sizing:content-box';}
  const botones=document.createElement('div');botones.style='display:flex;flex-wrap:wrap;gap:12px';root.appendChild(botones);
  function boton(t){const b=document.createElement('button');b.innerText=t;b.style='padding:10px 14px;font-size:15px;font-weight:bold;border-radius:7px;border:none;cursor:pointer;';botones.appendChild(b);return b;}
  const undo=boton('↶ Deshacer');const reset=boton('↺ Reiniciar');const finish=boton('✅ Terminar y guardar puntos');finish.disabled=true;
  document.body.appendChild(root);
  const vi=new Image(),pi=new Image();let pares=[],pendiente=null,terminado=false;
  function marcador(ctx,x,y,n){ctx.beginPath();ctx.arc(x,y,7,0,Math.PI*2);ctx.fillStyle='#ff185b';ctx.fill();ctx.strokeStyle='#fff';ctx.lineWidth=2;ctx.stroke();ctx.font='bold 24px Arial';ctx.lineWidth=3;ctx.strokeStyle='#111';ctx.strokeText(String(n),x+9,y-9);ctx.fillStyle='#fff';ctx.fillText(String(n),x+9,y-9);}
  function render(){
    if(vi.complete && vi.naturalWidth && cv.width){const ctx=cv.getContext('2d');ctx.clearRect(0,0,cv.width,cv.height);ctx.drawImage(vi,0,0);pares.forEach((p,i)=>marcador(ctx,p.pixel[0]*cv.width/ORIGINAL_W,p.pixel[1]*cv.height/ORIGINAL_H,i+1));if(pendiente)marcador(ctx,pendiente[0]*cv.width/ORIGINAL_W,pendiente[1]*cv.height/ORIGINAL_H,pares.length+1);}
    if(pi.complete && pi.naturalWidth && cp.width){const ctx=cp.getContext('2d');ctx.clearRect(0,0,cp.width,cp.height);ctx.drawImage(pi,0,0);pares.forEach((p,i)=>marcador(ctx,PAD+p.field[0]*SCALE,PAD+p.field[1]*SCALE,i+1));}
    finish.disabled=pares.length<4 || pendiente!==null || terminado;
    if(terminado){estado.textContent='✅ ¡Puntos guardados! Continúa a la siguiente celda.';return;}
    if(pendiente){estado.textContent='Pares completos: '+pares.length+' | 🟢 AHORA haz clic en el mismo punto del MAPA INFERIOR.';}
    else{estado.textContent='Pares completos: '+pares.length+' | 🔴 AHORA haz clic en una intersección del VIDEO SUPERIOR.';}
  }
  function pos(e,c){const r=c.getBoundingClientRect();return [(e.clientX-r.left)*c.width/r.width,(e.clientY-r.top)*c.height/r.height];}
  cv.addEventListener('click',e=>{if(pendiente||terminado||!cv.width)return;const [x,y]=pos(e,cv);pendiente=[x*ORIGINAL_W/cv.width,y*ORIGINAL_H/cv.height];render();});
  cp.addEventListener('click',e=>{if(!pendiente||terminado||!cp.width)return;const [x,y]=pos(e,cp);let xm=(x-PAD)/SCALE,ym=(y-PAD)/SCALE;if(xm<0||xm>LARGO||ym<0||ym>ANCHO){alert('Marca un lugar dentro del rectángulo de juego.');return;}pares.push({pixel:pendiente,field:[xm,ym]});pendiente=null;render();});
  undo.addEventListener('click',()=>{if(terminado)return;pendiente=null;pares.pop();render();});
  reset.addEventListener('click',()=>{if(terminado)return;pares=[];pendiente=null;render();});
  finish.addEventListener('click',()=>{if(pares.length>=4&&!pendiente&&!terminado){terminado=true;render();resolve(pares);}});
  vi.onload=()=>{cv.width=vi.naturalWidth;cv.height=vi.naturalHeight;render();};
  pi.onload=()=>{cp.width=pi.naturalWidth;cp.height=pi.naturalHeight;render();};
  vi.onerror=()=>{estado.textContent='❌ No se pudo cargar la imagen del video';reject(new Error('Video no cargó'));};
  pi.onerror=()=>{estado.textContent='❌ No se pudo cargar el plano';reject(new Error('Plano no cargó'));};
  vi.src=VIDEO_URL;pi.src=PITCH_URL;
  render();
})"""
for key,val in {
    '__URL_FRAME__':url_frame,'__URL_PLANO__':url_plano,
    '__ORIGINAL_W__':str(ancho_original),'__ORIGINAL_H__':str(alto_original),
    '__PAD__':str(MARGEN_PLANO),'__SCALE__':str(ESCALA_PLANO),
    '__LARGO__':str(LARGO_M),'__ANCHO__':str(ANCHO_M)
}.items():
    js=js.replace(key,val)
PARES = eval_js(js,timeout_sec=1800)
print(f'Capturados {len(PARES)} pares de puntos.')
for i,p in enumerate(PARES,1):print(i,'pixel:',[round(v,1) for v in p['pixel']],'campo (m):',[round(v,2) for v in p['field']])

## Celda 4 — Calcular la homografía

Ejecuta esta celda **solo después** de pulsar «Terminar y guardar puntos». Imprime la matriz `H` y los errores de la calibración. Con cuatro referencias el error puede ser cero aunque hayas marcado mal los puntos: para validar usa una referencia adicional independiente.

In [ ]:
# Matriz H: transforma (pixel_x, pixel_y) -> (metros_x, metros_y).
if len(PARES)<4:
    raise ValueError('Se necesitan >=4 pares de referencias.')
pixeles=np.asarray([p['pixel'] for p in PARES],dtype=np.float64)
metros=np.asarray([p['field'] for p in PARES],dtype=np.float64)

if cv2.contourArea(cv2.convexHull(pixeles.astype('float32')))<100:
    raise ValueError('Puntos del fotograma demasiado alineados o muy juntos.')
if cv2.contourArea(cv2.convexHull(metros.astype('float32')))<1:
    raise ValueError('Puntos de cancha demasiado alineados o juntos.')

metodo=cv2.RANSAC if len(PARES)>=5 else 0
H, mascara=cv2.findHomography(pixeles,metros,metodo, ransacReprojThreshold=1.5)
if H is None or not np.isfinite(H).all() or abs(np.linalg.det(H))<1e-12:
    raise ValueError('No se pudo determinar una homografía estable. Elige más puntos, no colineales.')

proyeccion=cv2.perspectiveTransform(pixeles.astype('float32').reshape(-1,1,2),H).reshape(-1,2)
errores=np.linalg.norm(proyeccion-metros,axis=1)
print('H (píxeles -> metros):')
print(H)
print('Error de ajuste por referencia (m):',np.round(errores,2))
print('RMSE de ajuste (m):',round(float(np.sqrt(np.mean(errores**2))),3))
if len(PARES)==4:
    print('AVISO: con solo 4 puntos, el error de ajuste puede ser casi cero incluso con referencias equivocadas.')
    print('Valida usando AL MENOS un quinto landmark conocido que no se usó para calibrar.')
else:
    print('Inliers RANSAC:',int(mascara.sum()),'/',len(PARES))

# Prueba más honesta cuando hay >=5 referencias: reservar la última como punto independiente.
if len(PARES)>=5:
    H_test,_=cv2.findHomography(pixeles[:-1],metros[:-1], cv2.RANSAC if len(PARES)>5 else 0, ransacReprojThreshold=1.5)
    if H_test is not None:
        est=cv2.perspectiveTransform(pixeles[-1:].astype('float32').reshape(-1,1,2),H_test).reshape(-1,2)[0]
        error_test=float(np.linalg.norm(est-metros[-1]))
        print(f'Error independiente del último landmark: {error_test:.2f} m')
        if error_test>2:
            print('ALERTA: revisar correspondencias. Error independiente >2 m (umbral práctico, no norma universal).')

## Celda 5 — Mostrar jugadores en el mapa

Si los puntos fueron correctos, verás algunos jugadores proyectados en el plano verde. Revisa que aparezcan en zonas lógicas; si no coinciden, repite la selección.

In [ ]:
# Proyectar los jugadores del mismo fotograma de calibración.
tracks_frame=tracks[tracks.frame==frame_idx].copy()
if tracks_frame.empty:
    raise ValueError('Este frame no aparece en el CSV. Verifica FPS, video y CSV.')

# Punto de contacto aproximado con el suelo = centro inferior de la bounding box
pies_px=np.column_stack(((tracks_frame.x1.to_numpy()+tracks_frame.x2.to_numpy())/2,
                          tracks_frame.y2.to_numpy())).astype('float32')
pies_m=cv2.perspectiveTransform(pies_px.reshape(-1,1,2),H).reshape(-1,2)
tracks_frame['x_m']=pies_m[:,0]
tracks_frame['y_m']=pies_m[:,1]
tracks_frame['en_cancha']=(tracks_frame.x_m.between(0,LARGO_M) & tracks_frame.y_m.between(0,ANCHO_M))

# Plano con jugadores, dentro del campo.
img=plano_bgr.copy()
for p in tracks_frame.itertuples():
    if not p.en_cancha:continue
    x,y=int(round(MARGEN_PLANO+p.x_m*ESCALA_PLANO)),int(round(MARGEN_PLANO+p.y_m*ESCALA_PLANO))
    cv2.circle(img,(x,y),5,(35,45,255),-1)
    cv2.putText(img,str(p.track_id),(x+5,y-5),cv2.FONT_HERSHEY_SIMPLEX,0.42,(255,255,255),1,cv2.LINE_AA)

fig,ax=plt.subplots(1,2,figsize=(18,7))
frame_anotado=frame_bgr.copy()
for p in tracks_frame.itertuples():
    foot=(int(round((p.x1+p.x2)/2)),int(p.y2))
    cv2.circle(frame_anotado,foot,4,(0,255,255),-1)
ax[0].imshow(cv2.cvtColor(frame_anotado,cv2.COLOR_BGR2RGB))
ax[0].set_title('Pies aproximados: mismo frame de calibración');ax[0].axis('off')
ax[1].imshow(cv2.cvtColor(img,cv2.COLOR_BGR2RGB))
ax[1].set_title('Proyección en metros | puntos fuera del césped omitidos');ax[1].axis('off')
plt.tight_layout();plt.show()

print('Tracks visibles en el frame:',len(tracks_frame))
print('Puntos proyectados dentro de la cancha:',int(tracks_frame.en_cancha.sum()))
print(tracks_frame[['track_id','x_m','y_m','en_cancha']].round(2).to_string(index=False))

## Celda 6 — Guardar resultados

Exporta la matriz de homografía en `.json`, un `.csv` con coordenadas de ese fotograma y un `.png` del mapa con jugadores.

In [ ]:
import json
nombre=VIDEO.stem
path_json=SALIDA/f'{nombre}_homografia.json'
path_csv=SALIDA/f'{nombre}_posiciones_frame_{frame_idx}.csv'
path_png=SALIDA/f'{nombre}_minimapa_frame_{frame_idx}.png'

info={
  'video':str(VIDEO),
  'csv_de_tracks':str(CSV_TRACKS),
  'fps':fps,
  'frame_calibrado':int(frame_idx),
  'tiempo_calibrado_s':frame_idx/fps,
  'largo_cancha_m':LARGO_M,
  'ancho_cancha_m':ANCHO_M,
  'H_pixel_a_metros':H.tolist(),
  'referencias':PARES,
  'rmse_ajuste_m':float(np.sqrt(np.mean(errores**2))),
  'advertencia':'H VALIDA SOLO para el frame calibrado o un intervalo SIN movimiento apreciable de cámara.'
}
path_json.write_text(json.dumps(info,indent=2,ensure_ascii=False),encoding='utf-8')
tracks_frame.to_csv(path_csv,index=False)
cv2.imwrite(str(path_png),img)
print('Archivos creados:')
print(path_json)
print(path_csv)
print(path_png)
print('\nNO utilices una H fija para todo el clip si hay paneo, zoom o corte de cámara.')

## Siguiente fase de FUT.AI

Cuando confirmemos que los puntos del minimapa coinciden **con posiciones reales plausibles**:

1. Validar la homografía con uno o más landmarks no usados para su ajuste.
2. Para clips con cámara móvil, **estimar o actualizar H por fotograma o por segmento estable** con referencias de cancha (keypoints, líneas). Si no hay referencias suficientes, marcar el fotograma como *sin calibración fiable*.
3. Solo entonces generar series `(track_id, tiempo_s, x_m, y_m)` y calcular distancia/velocidad con manejo de IDs interrumpidos, saltos anómalos y tiempos reales.

**No confundas el control de cámara de BoT-SORT con calibración del campo:** GMC puede mejorar la asociación de identidades, pero no produce por sí solo posiciones en metros.